<a href="https://colab.research.google.com/github/arturexxx/Alura-AgenteIA-RAG-PDF/blob/main/notebooks/Agente_RETCC_Alura_Desafio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================================================================
# PROYECTO: AGENTE DE IA PARA CONSULTAS DEL REGISTRO NACIONAL DE TRABAJADORES DE CONSTRUCCIÓN CIVIL (RETCC)
# ==========================================================================================================
# Descripción:
# Este proyecto implementa un sistema RAG (Retrieval-Augmented
# Generation) capaz de comprender el contenido de múltiples
# documentos PDF relacionados con el Registro Nacional de
# Trabajadores de Construcción Civil (RETCC) en el Estado Peruano.
#
# El agente podrá responder preguntas sobre:
# - Reglamento del RETCC.
# - Requisitos para la inscripción del Carné RETCC.
#
# Documentos utilizados:
# 1. Reglamento_Inscripcion.pdf
# 2. Requisitos_Inscripcion_Carnet_RETCC.pdf
#
# Tecnologías:
# - Python
# - Google Colab
# - LangChain
# - Hugging Face (Embeddings)
# - FAISS (Base Vectorial)
# - Google Gemini (LLM)
#
# Objetivo:
# Construir un asistente inteligente que consulte la información
# de los documentos y responda preguntas utilizando RAG.
# ============================================================

print("🚀 Iniciando el proyecto: Agente IA RAG para documentos del RETCC")

🚀 Iniciando el proyecto: Agente IA RAG para documentos del RETCC


In [2]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-google-genai \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    python-dotenv \
    gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
from google.colab import files

archivos_subidos = files.upload()

Saving Requisitos_Inscripcion_Carnet_RETCC.pdf to Requisitos_Inscripcion_Carnet_RETCC.pdf
Saving Reglamento_Incripcion.pdf to Reglamento_Incripcion.pdf


In [4]:
lista_pdfs = list(archivos_subidos.keys())

print(lista_pdfs)

['Requisitos_Inscripcion_Carnet_RETCC.pdf', 'Reglamento_Incripcion.pdf']


In [5]:
from langchain_community.document_loaders import PyPDFLoader

documentos = []

for pdf in lista_pdfs:
    print(f"Cargando: {pdf}")

    loader = PyPDFLoader(pdf)
    docs = loader.load()

    documentos.extend(docs)

print("\n-----------------------------")
print(f"Total de páginas cargadas: {len(documentos)}")

/tmp/ipykernel_1165/3316716.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Cargando: Requisitos_Inscripcion_Carnet_RETCC.pdf
Cargando: Reglamento_Incripcion.pdf

-----------------------------
Total de páginas cargadas: 9


In [6]:
import re

def limpiar_texto(texto: str) -> str:
    """
    Limpia el texto extraído del PDF sin eliminar información importante.
    """

    # Reemplaza espacios especiales
    texto = texto.replace("\xa0", " ")

    # Une palabras separadas por guion al final de una línea
    # Ejemplo: "traba-\njador" -> "trabajador"
    texto = re.sub(r"(\w)-\n(\w)", r"\1\2", texto)

    # Reemplaza saltos simples de línea por espacios
    texto = re.sub(r"(?<!\n)\n(?!\n)", " ", texto)

    # Reduce múltiples saltos de línea
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    # Reduce múltiples espacios
    texto = re.sub(r"[ \t]+", " ", texto)

    # Elimina espacios al inicio y final
    return texto.strip()


print("Función de limpieza creada correctamente.")

Función de limpieza creada correctamente.


In [7]:
documentos_limpios = []

for documento in documentos:
    documento.page_content = limpiar_texto(documento.page_content)
    documentos_limpios.append(documento)

print(f"Páginas limpiadas: {len(documentos_limpios)}")

Páginas limpiadas: 9


In [8]:
print(documentos_limpios[0].page_content[:2000])

Servicios del RETCC en las sedes del MAC de Lima Metropolitana. + Desde el día 13 de octubre de 2020 se etectuado la reapertura de los servicios de las MAC de Lima Sur y Lima Este; asimismo con fecha el 21 de marzo del 2022 se iniciaron as atenciones en el centro MAC Lima Norte, cabe precisar que en aichos MAC se encuentra brindaco el servicio de RETCC, por parte del personal de la ¡Subarrección de Regssiros Generales (en adelante SDRG), pudiendo acercarse los acminisrados de manera presencial sin necesidad de sacar ca para realizar su trámite de inscripción o renovación o en caso contra a recoger su camé RETCC que haya sido aprobado de manera vitual Requisitos para la inscripción del carné de RETCC + Deberá extibirsu DNIfísico o su Hoja CA, asimismo deberá presentar su Cercado de Antecedentes Penales en caso de homonimia. Requisitos para la renovación del carné de RETCC: 1. Solicitud según formato, 2. Copia simple de cercado o constancia de capacitación emitida por SENCICO y otras ent

In [9]:
documentos_limpios = [
    documento
    for documento in documentos_limpios
    if documento.page_content.strip()
]

print(f"Páginas con contenido: {len(documentos_limpios)}")

Páginas con contenido: 9


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "; ",
        ", ",
        " ",
        ""
    ]
)

print("Divisor configurado correctamente.")

Divisor configurado correctamente.


In [11]:
fragmentos = text_splitter.split_documents(documentos_limpios)

print(f"Páginas procesadas: {len(documentos_limpios)}")
print(f"Fragmentos generados: {len(fragmentos)}")

Páginas procesadas: 9
Fragmentos generados: 35


In [12]:
for indice, fragmento in enumerate(fragmentos):
    pagina_langchain = fragmento.metadata.get("page", 0)

    fragmento.metadata["chunk_id"] = indice
    fragmento.metadata["pagina_real"] = pagina_langchain + 1

print("Metadatos de trazabilidad agregados.")

Metadatos de trazabilidad agregados.


In [13]:
cantidad_mostrar = min(5, len(fragmentos))

for indice in range(cantidad_mostrar):
    fragmento = fragmentos[indice]

    print("=" * 80)
    print(f"Fragmento: {fragmento.metadata['chunk_id']}")
    print(f"Página: {fragmento.metadata['pagina_real']}")
    print(f"Caracteres: {len(fragmento.page_content)}")
    print("-" * 80)
    print(fragmento.page_content[:700])
    print()

Fragmento: 0
Página: 1
Caracteres: 914
--------------------------------------------------------------------------------
Servicios del RETCC en las sedes del MAC de Lima Metropolitana. + Desde el día 13 de octubre de 2020 se etectuado la reapertura de los servicios de las MAC de Lima Sur y Lima Este; asimismo con fecha el 21 de marzo del 2022 se iniciaron as atenciones en el centro MAC Lima Norte, cabe precisar que en aichos MAC se encuentra brindaco el servicio de RETCC, por parte del personal de la ¡Subarrección de Regssiros Generales (en adelante SDRG), pudiendo acercarse los acminisrados de manera presencial sin necesidad de sacar ca para realizar su trámite de inscripción o renovación o en caso contra a recoger su camé RETCC que haya sido aprobado de manera vitual Requisitos para la inscripción del carné 

Fragmento: 1
Página: 1
Caracteres: 844
--------------------------------------------------------------------------------
. Requisitos para la renovación del carné de RETCC: 1. Sol

In [14]:
fragmentos_vacios = [
    fragmento
    for fragmento in fragmentos
    if not fragmento.page_content.strip()
]

longitudes = [
    len(fragmento.page_content)
    for fragmento in fragmentos
]

print(f"Total de fragmentos: {len(fragmentos)}")
print(f"Fragmentos vacíos: {len(fragmentos_vacios)}")

if longitudes:
    print(f"Tamaño mínimo: {min(longitudes)} caracteres")
    print(f"Tamaño máximo: {max(longitudes)} caracteres")
    print(f"Tamaño promedio: {sum(longitudes) / len(longitudes):.2f} caracteres")

Total de fragmentos: 35
Fragmentos vacíos: 0
Tamaño mínimo: 13 caracteres
Tamaño máximo: 1000 caracteres
Tamaño promedio: 795.03 caracteres


In [15]:
!pip install -qU langchain-huggingface sentence-transformers

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

print("HuggingFaceEmbeddings importado correctamente.")

HuggingFaceEmbeddings importado correctamente.


In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

In [18]:
MODELO_EMBEDDINGS = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print(f"Modelo seleccionado: {MODELO_EMBEDDINGS}")

Modelo seleccionado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [20]:
import torch

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Dispositivo seleccionado: {dispositivo}")

if dispositivo == "cuda":
    print(f"GPU disponible: {torch.cuda.get_device_name(0)}")
else:
    print("Se utilizará CPU.")

Dispositivo seleccionado: cpu
Se utilizará CPU.


In [21]:
modelo_embeddings = HuggingFaceEmbeddings(
    model_name=MODELO_EMBEDDINGS,
    model_kwargs={
        "device": dispositivo
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32
    }
)

print("Modelo de embeddings cargado correctamente.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo de embeddings cargado correctamente.


In [22]:
texto_prueba = (
    "¿Cuáles son los requisitos para inscribirse en el registro de trabajadores?"
)

vector_prueba = modelo_embeddings.embed_query(texto_prueba)

print(f"Texto procesado: {texto_prueba}")
print(f"Dimensión del vector: {len(vector_prueba)}")
print(f"Primeros 10 valores:\n{vector_prueba[:10]}")

Texto procesado: ¿Cuáles son los requisitos para inscribirse en el registro de trabajadores?
Dimensión del vector: 384
Primeros 10 valores:
[0.05956784263253212, 0.043882355093955994, -0.04187829792499542, 0.016272101551294327, -0.03363706171512604, 0.07336026430130005, -0.04705607518553734, -0.053295962512493134, -0.05444514751434326, 0.012277219444513321]


In [23]:
print(f"Cantidad de fragmentos disponibles: {len(fragmentos)}")

if not fragmentos:
    raise ValueError(
        "No existen fragmentos. Ejecuta primero la etapa "
        "de limpieza y división de documentos."
    )

Cantidad de fragmentos disponibles: 35


In [24]:
textos_fragmentos = [
    fragmento.page_content
    for fragmento in fragmentos
]

print(f"Textos preparados: {len(textos_fragmentos)}")

Textos preparados: 35


In [25]:
vectores_fragmentos = modelo_embeddings.embed_documents(
    textos_fragmentos
)

print("Embeddings generados correctamente.")
print(f"Cantidad de vectores: {len(vectores_fragmentos)}")
print(f"Dimensión de cada vector: {len(vectores_fragmentos[0])}")

Embeddings generados correctamente.
Cantidad de vectores: 35
Dimensión de cada vector: 384


In [26]:
cantidad_fragmentos = len(fragmentos)
cantidad_vectores = len(vectores_fragmentos)

print(f"Fragmentos: {cantidad_fragmentos}")
print(f"Vectores: {cantidad_vectores}")

assert cantidad_fragmentos == cantidad_vectores, (
    "La cantidad de fragmentos no coincide "
    "con la cantidad de vectores."
)

print("Validación correcta: cada fragmento tiene un vector.")

Fragmentos: 35
Vectores: 35
Validación correcta: cada fragmento tiene un vector.


In [27]:
indice = 0

print("DOCUMENTO:")
print(fragmentos[indice].metadata.get("source"))

print("\nPÁGINA:")
print(fragmentos[indice].metadata.get("pagina_real"))

print("\nTEXTO:")
print(fragmentos[indice].page_content[:500])

print("\nDIMENSIÓN DEL VECTOR:")
print(len(vectores_fragmentos[indice]))

print("\nPRIMEROS 10 VALORES:")
print(vectores_fragmentos[indice][:10])

DOCUMENTO:
Requisitos_Inscripcion_Carnet_RETCC.pdf

PÁGINA:
1

TEXTO:
Servicios del RETCC en las sedes del MAC de Lima Metropolitana. + Desde el día 13 de octubre de 2020 se etectuado la reapertura de los servicios de las MAC de Lima Sur y Lima Este; asimismo con fecha el 21 de marzo del 2022 se iniciaron as atenciones en el centro MAC Lima Norte, cabe precisar que en aichos MAC se encuentra brindaco el servicio de RETCC, por parte del personal de la ¡Subarrección de Regssiros Generales (en adelante SDRG), pudiendo acercarse los acminisrados de manera presencial s

DIMENSIÓN DEL VECTOR:
384

PRIMEROS 10 VALORES:
[0.06203269958496094, -0.11673278361558914, 0.06817048788070679, 0.02668732777237892, 0.015333595685660839, -0.017674658447504044, -0.10165576636791229, 0.035474635660648346, -0.017937272787094116, 0.0315062515437603]


In [28]:
textos_prueba = [
    "Requisitos necesarios para obtener el carné RETCC.",
    "Documentos que debe presentar el trabajador.",
    "El clima de Lima durante el verano."
]

vectores_prueba = modelo_embeddings.embed_documents(
    textos_prueba
)

print(f"Vectores de prueba generados: {len(vectores_prueba)}")

Vectores de prueba generados: 3


In [29]:
import numpy as np

def similitud_coseno(vector_a, vector_b):
    vector_a = np.array(vector_a)
    vector_b = np.array(vector_b)

    return np.dot(vector_a, vector_b) / (
        np.linalg.norm(vector_a) *
        np.linalg.norm(vector_b)
    )

In [30]:
similitud_relacionada = similitud_coseno(
    vectores_prueba[0],
    vectores_prueba[1]
)

similitud_no_relacionada = similitud_coseno(
    vectores_prueba[0],
    vectores_prueba[2]
)

print(
    "Similitud entre textos relacionados:",
    round(float(similitud_relacionada), 4)
)

print(
    "Similitud entre textos no relacionados:",
    round(float(similitud_no_relacionada), 4)
)

Similitud entre textos relacionados: 0.3759
Similitud entre textos no relacionados: 0.0421


In [31]:
from langchain_community.vectorstores import FAISS

print("FAISS importado correctamente.")

FAISS importado correctamente.


In [32]:
vectorstore = FAISS.from_documents(
    documents=fragmentos,
    embedding=modelo_embeddings
)

print("Base vectorial creada correctamente.")

Base vectorial creada correctamente.


In [33]:
print(type(vectorstore))

<class 'langchain_community.vectorstores.faiss.FAISS'>


In [34]:
pregunta = "¿Cuáles son los requisitos para obtener el carné RETCC?"

In [37]:
resultados = vectorstore.similarity_search(
    pregunta,
    k=3
)

In [38]:
for i, doc in enumerate(resultados, start=1):

    print("="*80)
    print(f"Resultado {i}")
    print(f"Archivo : {doc.metadata.get('source')}")
    print(f"Página  : {doc.metadata.get('pagina_real')}")
    print("-"*80)
    print(doc.page_content[:700])
    print()

Resultado 1
Archivo : Requisitos_Inscripcion_Carnet_RETCC.pdf
Página  : 1
--------------------------------------------------------------------------------
. Requisitos para la renovación del carné de RETCC: 1. Solicitud según formato, 2. Copia simple de cercado o constancia de capacitación emitida por SENCICO y otras entidades públicas o privadas habitadas por el sector educación o cerficar las competencias laborales en la aclvidad de construcción cv 2.1. Ceníicado o constancia de capacitación emitida por SENCICO y otras entidades públicas o pacas habitadas por el sector educación a) Condiciones dela capacitación -Para las categorías OFICIAL y OPERARIO, los ceríicados deben reunir un minimo de 32 horas de capacitación, puaiendo ser horas acumulables =Parala categoría de PEON. se requiere un minimo de 16 horas de capacitación. puglendo ser hora

Resultado 2
Archivo : Reglamento_Incripcion.pdf
Página  : 1
--------------------------------------------------------------------------------
. 

In [39]:
preguntas = [
    "¿Qué documentos necesito presentar?",
    "¿Cuánto dura el trámite?",
    "¿Cómo puedo renovar mi carné?",
    "¿Qué requisitos existen para la inscripción?"
]

In [40]:
for pregunta in preguntas:

    print("="*90)
    print("PREGUNTA:", pregunta)

    docs = vectorstore.similarity_search(
        pregunta,
        k=2
    )

    for doc in docs:

        print("\nDocumento:", doc.metadata["source"])
        print("Página:", doc.metadata["pagina_real"])
        print(doc.page_content[:250])

PREGUNTA: ¿Qué documentos necesito presentar?

Documento: Reglamento_Incripcion.pdf
Página: 1
. Artículo 6.- Requisitos para la Inscripción Para la inscripción en el RETCC, los trabajadores interesados deben presentar una solicitud según formato aprobado mediante resolución ministerial, adjuntando la siguiente documentación: a) Certificado o 

Documento: Reglamento_Incripcion.pdf
Página: 4
. Tercera.- Normas complementarias Mediante resolución ministerial se establece las disposiciones y medidas complementarias para la mejor aplicación del presente decreto supremo; entre otros aspectos para: a) La aprobación de los instrumentos técnico
PREGUNTA: ¿Cuánto dura el trámite?

Documento: Reglamento_Incripcion.pdf
Página: 3
vigencia de dos (02) años; pudiendo ser renovada desde los cuarenta y cinco (45) días calendario anteriores a la fecha de su vencimiento, hasta el último día de su vigencia. 10.2. De no renovarse oportunamente la inscripción, ésta caducará de manera 

Documento: Requisitos